# 00 — Stream health

Is the stream alive, and is it complete? Three separate questions, answered
separately: what is in the log, what is arriving now, and what went missing
in between.

**Before running:** you need a broker with data in it.

- Local: `make stream-local` in another terminal (brings up compose Kafka,
  creates the topics, runs the Binance + Coinbase producers on the host).
- MSK: `make up`, then set `FDAI_TARGET=msk` — or call
  `devlab.from_terraform()` below, which reads the endpoint straight from the
  stack outputs and cannot go stale.

In [ ]:
import devlab
from devlab import frames, health

target = devlab.resolve()  # $FDAI_TARGET: "local" (default), "msk", or "terraform"
target

## What is in the log

`messages` is `high - low` per partition, summed — what is **currently
retained**, not what was ever written. `md.trades.v1` ages out at 24h by
design, so this number plateaus rather than growing forever.

A topic at 0 is normal for everything except `md.trades.v1`: no other
producer code exists yet (see `docs/ARCHITECTURE.md`).

In [ ]:
frames.frame(devlab.topics(target))

## Is the keying working

Trades are keyed `venue|venue_symbol` so one instrument on one venue stays
ordered on one partition. With 8 instruments across 2 venues over 6
partitions, expect *uneven but non-empty* partitions. A single partition
holding everything means the key is not being set.

In [ ]:
frames.frame(devlab.partitions(target, "md.trades.v1"))

## Is anything arriving right now

Reads from `latest`, so this measures live arrivals only. Zero here with a
non-zero count above means the producers stopped — not that the topic is
empty.

In [ ]:
report = devlab.rate(target, seconds=10.0)
print(f"{report.messages} trades in {report.seconds:.1f}s = {report.per_second:.1f}/s")
report.by_venue

In [ ]:
frames.frame([{"instrument_id": k, "trades": v} for k, v in report.by_instrument.items()])

## Did anything go missing

Replays sequence numbers through `SequenceTracker` — the same detector the
producer runs inline.

Two caveats worth keeping in mind:

- **Coinbase is skipped, not reported clean.** Its `sequence_num` is
  connection-wide while the topic is partitioned by symbol, so any gap found
  here would be an artefact of reading across partitions.
- **A gap here is weaker evidence than one from `IngestRunner`.** It means the
  record never reached Kafka *or* is no longer retained — and the runner may
  already have repaired it via REST, landing it at a later offset.

In [ ]:
recent = devlab.collect(target, limit=5_000, seconds=30.0, offset_reset="earliest")
gaps = health.sequence_gaps(recent)

print(f"checked   {gaps.checked} sequenced records")
print(f"gaps      {len(gaps.gaps)} ({gaps.missing} trades missing)")
print(f"skipped   {gaps.skipped_venues or 'none'} (connection-scoped sequence)")

frames.frame(gaps.gaps) if gaps.gaps else "no gaps found"

## Consumer lag

For a named group — e.g. once a Databricks Bronze reader is running. `devlab`'s
own reads use random group ids with auto-commit off, so they never appear here
and never move anyone else's offsets.

`committed = None` means the group has never committed that partition.

In [ ]:
frames.frame(health.lag(target, group="fdai-bronze", topic="md.trades.v1"))